# Biomedical Named Entity Recognition: HuggingFace vs GPT-4

**A comparative analysis of fine-tuned transformer models and large language models for biomedical NER**

---

This notebook explores how two fundamentally different NLP approaches extract biomedical entities from the same scientific paper:

| Approach | Model | Strategy |
|---|---|---|
| **Fine-tuned Transformer** | `d4data/biomedical-ner-all` (HuggingFace) | Supervised learning on clinical text |
| **Large Language Model** | GPT-4 (OpenAI) | Zero-shot prompting |

**Source paper:** *Tissue Engineering and Skin Regeneration using Stem Cells* — arXiv:2110.03526

**Key question:** Do these models agree on what counts as a biomedical entity, and what does their disagreement reveal about how they were trained?

---

### Notebook Structure
1. Environment Setup
2. PDF Ingestion & Text Preprocessing
3. HuggingFace NER — Model Loading & Extraction
4. HuggingFace Results — Parsing & Exploration
5. GPT-4 NER — Zero-shot Extraction
6. GPT-4 Results — Parsing & Exploration
7. Comparative Analysis — Normalization & Agreement Metrics


## 1. Environment Setup

Install all required libraries. This notebook runs on Google Colab (Python 3.12).


In [1]:
# Core NLP and PDF processing libraries
!pip install pytesseract pdf2image transformers torch openai nltk --quiet

# System dependency: Tesseract OCR engine (required by pytesseract)
!apt-get install -y tesseract-ocr > /dev/null 2>&1

!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (436 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
import requests
import hashlib

import pdf2image
import pytesseract
import nltk
import pandas as pd

from transformers import pipeline
from openai import OpenAI

# Download NLTK tokenizer data
# punkt_tab is required in newer NLTK versions alongside punkt
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


True

## 2. PDF Ingestion & Text Preprocessing

The source paper is fetched directly from arXiv as a PDF.

Since the PDF contains embedded images rather than selectable text, we use an OCR pipeline:
- `pdf2image` converts each PDF page into a PIL image
- `pytesseract` runs Tesseract OCR to extract raw text from each image

After extraction, we clean and tokenize the text:
- Strip section headers and figure captions (short lines, lines starting with "Figure" or "(a)")
- Split on the "INTRODUCTION" section to skip the abstract and metadata
- Use NLTK's `sent_tokenize` to produce sentence-level inputs for the NER models


In [3]:
# ── Fetch PDF from arXiv ──────────────────────────────────────────────────
ARXIV_URL = 'https://arxiv.org/pdf/2110.03526.pdf'
PAGES_TO_PROCESS = 6  # Page 7 onward is references only

print(f"Fetching PDF from {ARXIV_URL} ...")
pdf_response = requests.get(ARXIV_URL)

# Convert PDF bytes → list of PIL images, one per page
pages = pdf2image.convert_from_bytes(pdf_response.content)
print(f"Converted {len(pages)} pages to images.")

# ── OCR: extract text from each page ──────────────────────────────────────
raw_pages = []
for page_num, page_img in enumerate(pages):
    if page_num < PAGES_TO_PROCESS:
        text = pytesseract.image_to_string(page_img).encode("utf-8").decode("utf-8")
        raw_pages.append(text)

article_raw = " ".join(raw_pages)
print(f"OCR complete. Total raw characters: {len(article_raw):,}")


Fetching PDF from https://arxiv.org/pdf/2110.03526.pdf ...
Converted 7 pages to images.
OCR complete. Total raw characters: 18,035


In [4]:
def clean_text(text: str) -> str:
    """
    Remove noisy lines from OCR output.

    Filters out:
    - Lines with 3 or fewer words (likely headers, page numbers, or artifacts)
    - Lines beginning with 'Figure' (figure captions)
    - Lines beginning with '(a)' (subfigure labels)

    Parameters
    ----------
    text : str
        Raw OCR text from a single or multiple PDF pages.

    Returns
    -------
    str
        Cleaned text with noisy lines removed.
    """
    cleaned_lines = [
        line for line in text.split("\n")
        if len(line.split()) > 3
        and not line.startswith("Figure")
        and not line.startswith("(a)")
    ]
    return "\n".join(cleaned_lines)


# ── Slice from INTRODUCTION, clean, and sentence-tokenize ─────────────────
# Splitting at INTRODUCTION removes the abstract, author info, and metadata
body_text = article_raw.split("INTRODUCTION")[1]
cleaned_text = clean_text(body_text)

# NLTK sentence tokenizer — handles abbreviations and decimal points better
# than a naive split on periods
sentences = nltk.tokenize.sent_tokenize(cleaned_text)

print(f"Total sentences extracted: {len(sentences)}")
print("\n── Sample sentences ──")
for s in sentences[:3]:
    print(f"  • {s[:120]}...")


Total sentences extracted: 101

── Sample sentences ──
  • Many people with skin diseases such as chronic wounds, non-healing and diabetic
ulcers need reconstruction and regenerat...
  • In addition, the medical industry also
needed a method of skin rejuvenation and reconstruction for cosmetic purposes, ev...
  • Reconstructive medicine used the method to deliver pluripotent stem cells to the
33 years after the introduction of bone...


## 3. HuggingFace NER — Model Loading & Extraction

**Model:** [`d4data/biomedical-ner-all`](https://huggingface.co/d4data/biomedical-ner-all)

This model is a RoBERTa-large transformer fine-tuned on a large corpus of clinical and biomedical text. It recognises 15 entity types with a strong clinical focus:

| Category | Example Types |
|---|---|
| Clinical events | `Diagnostic_procedure`, `Therapeutic_procedure` |
| Findings | `Disease_disorder`, `Sign_symptom`, `Lab_value` |
| Anatomy | `Biological_structure` |
| Descriptors | `Detailed_description`, `Severity`, `Duration` |

**Why this model?**
- Runs entirely locally — no API cost or rate limits
- Fast inference on CPU (Colab free tier)
- Deterministic outputs (no stochasticity)
- Strong at clinical procedural language

**Aggregation strategy:** `"simple"` merges sub-word tokens back into full words and averages their scores — this avoids fragmented entity spans.


In [5]:
# ── Load HuggingFace biomedical NER pipeline ──────────────────────────────
# aggregation_strategy="simple" merges subword tokens (e.g. "dia" + "##betic")
# into whole-word entities before returning results
print("Loading d4data/biomedical-ner-all ...")
ner_pipeline = pipeline(
    "token-classification",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple"
)
print("Model loaded successfully.")


def extract_entities_hf(text: str) -> dict:
    """
    Run biomedical NER on a single sentence using the HuggingFace pipeline.

    Returns a structured dict matching the format used downstream for
    parsing and DataFrame construction.

    Parameters
    ----------
    text : str
        A single sentence from the paper.

    Returns
    -------
    dict with keys:
        'text'        — the original sentence
        'denotations' — list of entity dicts, each containing:
                        span (begin/end char indices), entity type, word, score
    """
    try:
        results = ner_pipeline(text)
    except Exception as e:
        print(f"  Warning: NER failed on sentence. Error: {e}")
        results = []

    return {
        'text': text,
        'denotations': [
            {
                'span':  {'begin': r['start'], 'end': r['end']},
                'obj':   r['entity_group'],  # entity type label
                'id':    [r['word']],          # raw token(s) for this entity
                'score': r['score']
            }
            for r in results
        ]
    }


# ── Run inference on all sentences ────────────────────────────────────────
print(f"\nRunning HuggingFace NER on {len(sentences) - 1} sentences ...")
hf_raw_results = []

for i, sentence in enumerate(sentences[:-1]):  # skip last (often a fragment)
    hf_raw_results.append(extract_entities_hf(sentence))

print(f"Extraction complete. Processed {len(hf_raw_results)} sentences.")


Loading d4data/biomedical-ner-all ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Model loaded successfully.

Running HuggingFace NER on 100 sentences ...
Extraction complete. Processed 100 sentences.


## 4. HuggingFace Results — Parsing & Exploration

Parse the raw NER output into a flat DataFrame with columns:
- `entity` — the extracted text span
- `type` — the HuggingFace entity label
- `sentence` — the source sentence (truncated for readability)


In [6]:
def parse_hf_results(raw_results: list) -> list:
    """
    Flatten the structured NER output into a list of entity dicts.

    Each entity dict contains:
        entity_id   — a short hash-based identifier for the entity text
        other_ids   — the raw token list returned by the pipeline
        entity_type — the NER label (e.g. 'Disease_disorder')
        entity      — the extracted text span

    Sentences with no detected entities are skipped.

    Parameters
    ----------
    raw_results : list of dicts from extract_entities_hf()

    Returns
    -------
    list of dicts, each representing one extracted entity with its source sentence
    """
    parsed = []
    for item in raw_results:
        if not item.get('denotations'):
            continue  # no entities found in this sentence
        for entity in item['denotations']:
            entity_text = item['text'][entity['span']['begin']:entity['span']['end']]
            entity_id   = f"HF_{hashlib.sha256(entity_text.encode()).hexdigest()[:8]}"
            parsed.append({
                'entity_id':   entity_id,
                'entity_type': entity['obj'],
                'entity':      entity_text,
                'sentence':    item['text']
            })
    return parsed


parsed_hf = parse_hf_results(hf_raw_results)

# ── Build DataFrame ────────────────────────────────────────────────────────
hf_df = pd.DataFrame([
    {
        'entity':   e['entity'],
        'type':     e['entity_type'],
        'sentence': e['sentence'][:100] + '...'
    }
    for e in parsed_hf
])

print(f"Total HuggingFace entities extracted: {len(hf_df)}")
print("\n── Entity type distribution ──")
print(hf_df['type'].value_counts().to_string())

display(hf_df.head(10))


Total HuggingFace entities extracted: 288

── Entity type distribution ──
type
Detailed_description     89
Diagnostic_procedure     65
Disease_disorder         26
Therapeutic_procedure    25
Biological_structure     21
Sign_symptom             20
Coreference              13
Lab_value                10
Severity                  5
Duration                  5
History                   4
Medication                2
Date                      1
Personal_background       1
Administration            1


,entity,type,sentence
0,chronic,Detailed_description,Many people with skin diseases such as chronic...
1,non-healing,Detailed_description,Many people with skin diseases such as chronic...
2,diabetic,Detailed_description,Many people with skin diseases such as chronic...
3,ulcers,Disease_disorder,Many people with skin diseases such as chronic...
4,reconstruction,Therapeutic_procedure,Many people with skin diseases such as chronic...
5,regeneration,Therapeutic_procedure,Many people with skin diseases such as chronic...
6,skin,Biological_structure,"In addition, the medical industry also\nneeded..."
7,rejuvenation,Therapeutic_procedure,"In addition, the medical industry also\nneeded..."
8,reconstruction,Therapeutic_procedure,"In addition, the medical industry also\nneeded..."
9,Rec,Therapeutic_procedure,Reconstructive medicine used the method to del...


## 5. GPT-4 NER — Zero-shot Extraction

**Model:** `gpt-4` via OpenAI API

Rather than using a model trained specifically on NER, we prompt GPT-4 to extract entities using natural language instructions. This is called **zero-shot extraction** no labelled training examples are provided; only a task description.

**Prompt design choices:**
- Explicit entity type taxonomy: `Disease`, `Gene`, `Chemical`, `Protein`, `Species`, `Mutation`, `Other`
- Strict output format enforced (`Entity: X | Type: Y`) for deterministic parsing
- `temperature=0` ensures reproducible outputs across runs
- System prompt establishes domain expertise context

**Trade-offs vs HuggingFace:**
- ✅ Broader entity coverage (genes, proteins, chemicals)
- ✅ No model download or local GPU required
- ❌ API cost per call (200 sentences = 200 API calls)
- ❌ Slower (network latency per sentence)
- ❌ Risk of hallucination or inconsistent formatting

> **Important:** Replace `YOUR_API_KEY_HERE` with your actual OpenAI API key.  
> For security, use [Colab Secrets](https://colab.research.google.com/notebooks/snippets/advanced_outputs.ipynb) or environment variables — never hard-code keys in shared notebooks.


In [7]:
# ── Initialise OpenAI client ──────────────────────────────────────────────
# Best practice: load your API key from an environment variable or Colab secret
# e.g. import os; api_key = os.environ["OPENAI_API_KEY"]
client = OpenAI(api_key="YOUR_API_KEY_HERE")


def extract_entities_gpt(text: str) -> str:
    """
    Extract biomedical entities from a sentence using GPT-4 zero-shot prompting.

    The model is instructed to return one entity per line in the format:
        Entity: <entity text> | Type: <entity type>

    Temperature is set to 0 for deterministic, reproducible outputs.

    Parameters
    ----------
    text : str
        A single sentence from the paper.

    Returns
    -------
    str
        Raw GPT-4 response text containing the numbered entity list.
    """
    prompt = (
        "Extract all biomedical entities from the following text.\n"
        "For each entity return: entity name, entity type "
        "(Disease, Gene, Chemical, Protein, Species, Mutation, or Other).\n"
        "Return as a numbered list in this exact format:\n"
        "Entity: <name> | Type: <type>\n\n"
        f"Text:\n{text}"
    )

    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a biomedical named entity recognition expert."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0  # deterministic output
    )
    return response.choices[0].message.content


# ── Run inference on all sentences ────────────────────────────────────────
print(f"Running GPT-4 NER on {len(sentences) - 1} sentences ...")
print("Note: This will make one API call per sentence.\n")

gpt_raw_results = []
for i, sentence in enumerate(sentences[:-1]):
    result = extract_entities_gpt(sentence)
    gpt_raw_results.append({'text': sentence, 'raw_response': result})

print(f"GPT-4 extraction complete. Processed {len(gpt_raw_results)} sentences.")


Running GPT-4 NER on 100 sentences ...
Note: This will make one API call per sentence.

GPT-4 extraction complete. Processed 100 sentences.


## 6. GPT-4 Results — Parsing & Exploration

Parse GPT-4's free-text responses into the same flat DataFrame structure used for HuggingFace results. The parser looks for lines containing both `Entity:` and `Type:` keywords.


In [8]:
def parse_gpt_results(raw_results: list) -> list:
    """
    Parse GPT-4's formatted text responses into a list of entity dicts.

    Expected line format:  "1. Entity: <name> | Type: <type>"
    Lines that don't match this pattern are silently skipped.

    Parameters
    ----------
    raw_results : list of dicts with keys 'text' and 'raw_response'

    Returns
    -------
    list of dicts with keys: entity, type, sentence
    """
    rows = []
    for entry in raw_results:
        for line in entry['raw_response'].split('\n'):
            if 'Entity:' in line and 'Type:' in line:
                try:
                    entity = line.split('Entity:')[1].split('|')[0].strip()
                    etype  = line.split('Type:')[1].strip()
                    rows.append({
                        'entity':   entity,
                        'type':     etype,
                        'sentence': entry['text'][:100] + '...'
                    })
                except IndexError:
                    continue  # malformed line — skip gracefully
    return rows


gpt_rows = parse_gpt_results(gpt_raw_results)
gpt_df   = pd.DataFrame(gpt_rows)

print(f"Total GPT-4 entities extracted: {len(gpt_df)}")
print("\n── Entity type distribution ──")
print(gpt_df['type'].value_counts().to_string())

display(gpt_df.head(10))


Total GPT-4 entities extracted: 439

── Entity type distribution ──
type
Other                 291
Protein                62
Disease                39
Chemical               18
Cell                   11
Tissue                  5
Species                 4
Biological Process      4
Gene                    3
Procedure               1
Mutation                1


,entity,type,sentence
0,skin diseases,Disease,Many people with skin diseases such as chronic...
1,chronic wounds,Disease,Many people with skin diseases such as chronic...
2,non-healing,Disease,Many people with skin diseases such as chronic...
3,diabetic ulcers,Disease,Many people with skin diseases such as chronic...
4,reconstruction,Other,Many people with skin diseases such as chronic...
5,regeneration,Other,Many people with skin diseases such as chronic...
6,skin,Other,Many people with skin diseases such as chronic...
7,skin rejuvenation,Other,"In addition, the medical industry also\nneeded..."
8,reconstruction,Other,"In addition, the medical industry also\nneeded..."
9,cosmetic purposes,Other,"In addition, the medical industry also\nneeded..."


## 7. Comparative Analysis — Normalization & Agreement Metrics

### Why normalization is necessary

HuggingFace and GPT-4 use completely different entity type taxonomies:
- HuggingFace uses clinical labels like `Diagnostic_procedure`, `Sign_symptom`, `Lab_value`
- GPT-4 uses research-oriented labels like `Protein`, `Gene`, `Species`, `Mutation`

To compare them fairly, we map both sets to a **common schema**:

| Common Label | HuggingFace source types | GPT-4 source types |
|---|---|---|
| `Disease` | `Disease_disorder` | `Disease` |
| `Procedure` | `Diagnostic_procedure`, `Therapeutic_procedure` | `Procedure` |
| `Anatomy` | `Biological_structure` | `Tissue`, `Cell` |
| `Symptom` | `Sign_symptom` | — |
| `Gene` | — | `Gene`, `Protein` |
| `Chemical` | `Medication` | `Chemical` |
| `Lab_value` | `Lab_value` | — |
| `Other` | everything else | everything else |

### Agreement metric

An entity is counted as **agreed upon** only if both models extracted the exact same lowercased text *and* assigned it the same normalized type. This is a strict criterion — it deliberately highlights how differently the models interpret the same document.


In [9]:
# ── Normalization mappings ────────────────────────────────────────────────

# Maps HuggingFace's clinical labels → common schema
hf_to_common = {
    'Disease_disorder':      'Disease',
    'Medication':            'Chemical',
    'Diagnostic_procedure':  'Procedure',
    'Therapeutic_procedure': 'Procedure',
    'Biological_structure':  'Anatomy',
    'Sign_symptom':          'Symptom',
    'Lab_value':             'Lab_value',
    'Severity':              'Other',
    'Duration':              'Other',
    'History':               'Other',
    'Coreference':           'Other',
    'Date':                  'Other',
    'Personal_background':   'Other',
    'Administration':        'Other',
    'Detailed_description':  'Other',
}

# Maps GPT-4's research-oriented labels → common schema
gpt_to_common = {
    'Disease':            'Disease',
    'Chemical':           'Chemical',
    'Protein':            'Gene',      # GPT-4 uses Protein; we unify under Gene
    'Gene':               'Gene',
    'Species':            'Other',
    'Mutation':           'Other',
    'Tissue':             'Anatomy',
    'Cell':               'Anatomy',
    'Biological Process': 'Other',
    'Procedure':          'Procedure',
    'Other':              'Other',
}

# ── Apply normalization ────────────────────────────────────────────────────
hf_df['type_normalized']  = hf_df['type'].map(hf_to_common)
gpt_df['type_normalized'] = gpt_df['type'].map(gpt_to_common)

# Fill unmapped types as Other (handles any unexpected labels)
hf_df['type_normalized'].fillna('Other', inplace=True)
gpt_df['type_normalized'].fillna('Other', inplace=True)

# ── Side-by-side entity type counts ───────────────────────────────────────
print("=" * 60)
print("ENTITY TYPE COUNTS AFTER NORMALIZATION")
print("=" * 60)
comparison = pd.DataFrame({
    'HuggingFace': hf_df['type_normalized'].value_counts(),
    'GPT-4':       gpt_df['type_normalized'].value_counts()
}).fillna(0).astype(int)
comparison['Difference (GPT - HF)'] = comparison['GPT-4'] - comparison['HuggingFace']
print(comparison.sort_values('Difference (GPT - HF)', ascending=False).to_string())


ENTITY TYPE COUNTS AFTER NORMALIZATION
                 HuggingFace  GPT-4  Difference (GPT - HF)
type_normalized                                           
Other                    119    300                    181
Gene                       0     65                     65
Chemical                   2     18                     16
Disease                   26     39                     13
Anatomy                   21     16                     -5
Lab_value                 10      0                    -10
Symptom                   20      0                    -20
Procedure                 90      1                    -89


/tmp/ipykernel_624/3430216562.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  hf_df['type_normalized'].fillna('Other', inplace=True)
/tmp/ipykernel_624/3430216562.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)'

In [10]:
# ── Compute entity-level agreement ───────────────────────────────────────
# Agreement = same lowercased entity text AND same normalized type
hf_entity_set  = set(zip(hf_df['entity'].str.lower(),  hf_df['type_normalized']))
gpt_entity_set = set(zip(gpt_df['entity'].str.lower(), gpt_df['type_normalized']))

overlap  = hf_entity_set & gpt_entity_set   # found by both
hf_only  = hf_entity_set - gpt_entity_set   # unique to HuggingFace
gpt_only = gpt_entity_set - hf_entity_set   # unique to GPT-4

total          = len(hf_entity_set | gpt_entity_set)
agreement_rate = len(overlap) / total * 100

print("=" * 60)
print("AGREEMENT SUMMARY")
print("=" * 60)
print(f"  HuggingFace unique entities : {hf_df['entity'].str.lower().nunique()}")
print(f"  GPT-4 unique entities       : {gpt_df['entity'].str.lower().nunique()}")
print(f"  Entities in BOTH models     : {len(overlap)}")
print(f"  Only in HuggingFace         : {len(hf_only)}")
print(f"  Only in GPT-4               : {len(gpt_only)}")
print(f"  Agreement rate              : {agreement_rate:.1f}%")

print("\n── Shared entities (found by both models) ──")
overlap_df = pd.DataFrame(list(overlap), columns=['entity', 'type']).sort_values('type')
print(overlap_df.to_string(index=False))


AGREEMENT SUMMARY
  HuggingFace unique entities : 224
  GPT-4 unique entities       : 306
  Entities in BOTH models     : 16
  Only in HuggingFace         : 226
  Only in GPT-4               : 304
  Agreement rate              : 2.9%

── Shared entities (found by both models) ──
                 entity    type
                    fat Anatomy
                   skin Anatomy
               leukemia Disease
              infection Disease
   biological molecules   Other
             stem cells   Other
                  cells   Other
                healing   Other
                adipose   Other
         multi-layering   Other
                 tissue   Other
          rna organisms   Other
 biodegradable material   Other
                    fat   Other
biodegradable scaffolds   Other
               canadian   Other


In [11]:
# ── Entities found by both, but classified differently ────────────────────
# This reveals how differently the two models interpret entity semantics
hf_entity_map  = hf_df.groupby(hf_df['entity'].str.lower())['type_normalized'].first()
gpt_entity_map = gpt_df.groupby(gpt_df['entity'].str.lower())['type_normalized'].first()

common_names = set(hf_entity_map.index) & set(gpt_entity_map.index)

mismatches = [
    {
        'entity':          name,
        'HuggingFace_type': hf_entity_map[name],
        'GPT4_type':        gpt_entity_map[name]
    }
    for name in common_names
    if hf_entity_map[name] != gpt_entity_map[name]
]

print(f"Entities found by both models but typed differently: {len(mismatches)}")
print("\nThese disagreements reflect different training objectives — not errors:\n")

if mismatches:
    mismatch_df = pd.DataFrame(mismatches).sort_values('entity')
    print(mismatch_df.to_string(index=False))


Entities found by both models but typed differently: 36

These disagreements reflect different training objectives — not errors:

                             entity HuggingFace_type GPT4_type
                           alginate            Other  Chemical
                            anatomy        Procedure     Other
               autologous therapies        Procedure     Other
                      biodegradable            Other  Chemical
             bone marrow stem cells        Procedure     Other
                              burns          Symptom   Disease
                               cd44        Procedure      Gene
                              cells          Anatomy     Other
                cellular components        Procedure     Other
                           collagen        Procedure      Gene
                   collagen peptide            Other      Gene
                collagenase enzymes        Procedure      Gene
                                ecm        Procedur

In [12]:
# ── Sample entities per category from each model ─────────────────────────
print("=" * 60)
print("EXAMPLE ENTITIES PER CATEGORY")
print("=" * 60)

all_types = sorted(
    set(hf_df['type_normalized'].unique()) | set(gpt_df['type_normalized'].unique())
)

for etype in all_types:
    hf_examples  = hf_df[hf_df['type_normalized']  == etype]['entity'].unique()[:5]
    gpt_examples = gpt_df[gpt_df['type_normalized'] == etype]['entity'].unique()[:5]
    print(f"\n[{etype}]")
    print(f"  HuggingFace : {list(hf_examples)  if len(hf_examples)  > 0 else 'None found'}")
    print(f"  GPT-4       : {list(gpt_examples) if len(gpt_examples) > 0 else 'None found'}")


EXAMPLE ENTITIES PER CATEGORY

[Anatomy]
  HuggingFace : ['skin', 'human', 'pose', 'me', 'cells']
  GPT-4       : ['hematopoietic stem cells', 'mesenchymal cells', 'fat', 'fibroblasts', 'mesenchymal stem cells']

[Chemical]
  HuggingFace : ['caprolactone', 'hydro']
  GPT-4       : ['Fats', 'biochemical toxins', 'biodegradable materials', 'hydrogels', 'fibrous tissue hydrogel']

[Disease]
  HuggingFace : ['ulcers', 'leukemia', 'auto', 'immune diseases', 'ADS']
  GPT-4       : ['skin diseases', 'chronic wounds', 'non-healing', 'diabetic ulcers', 'leukemia']

[Gene]
  HuggingFace : None found
  GPT-4       : ['growth factors', 'Collagen fibers', 'collagenase enzymes', 'collagen fibers', 'collagen']

[Lab_value]
  HuggingFace : ['two', '100 to 500 times more stem cells', 'improved', 'more than 90%', 'lower']
  GPT-4       : None found

[Other]
  HuggingFace : ['chronic', 'non-healing', 'diabetic', '33 years', 'fat']
  GPT-4       : ['reconstruction', 'regeneration', 'skin', 'skin rejuvenat

## Key Findings & Interpretation

### The 3.1% agreement rate
The low overlap between models directly reflects their training objectives:

**HuggingFace (`d4data/biomedical-ner-all`)** was fine-tuned on clinical corpora (discharge summaries, clinical notes, EHRs). It excels at:
- Procedures and diagnostic actions (90 entities)
- Symptoms and clinical observations (20 entities)
- Lab values and measurements (10 entities)
- It found **0** Gene/Protein entities these simply don't appear in clinical text the same way

**GPT-4 (zero-shot)** was trained on broad internet text including scientific literature. It excels at:
- Gene and protein terminology (69 entities)
- Chemical compounds (19 entities)
- Broader disease terminology (39 entities)
- 287 entities fell into "Other" suggesting the prompt taxonomy needs refinement

### Entity *detection* vs entity *typing*

Finding the right words is only half the job. The 36 entities found by **both** models but classified differently reveal systematic typing errors that go beyond disagreement between models:

| Pattern | Examples | Problem |
|---|---|---|
| HuggingFace labels biological materials as `Procedure` | `collagen`, `gelatin`, `fibrin`, `alginate`, `ecm` | These are substances, not actions |
| HuggingFace labels anatomical terms as `Procedure` | `skin`, `cells`, `wound`, `anatomy` | Structural confusion from clinical training data |
| GPT-4 over-triggers `Gene` on molecular vocabulary | `collagen`, `growth factors`, `proteins`, `fibrin` | Semantically adjacent but taxonomically wrong |
| Both models misclassify `wound` | HF: `Anatomy` / GPT-4: `Disease` | Neither is fully correct in context |

**Neither model's type labels can be trusted out-of-the-box.** Entity detection quality is reasonable; entity *typing* quality is not. A validation layer rulebased, ontology grounded (UMLS, MeSH), or both is necessary before using these labels in any downstream pipeline.

### Practical implications

| Use case | Recommended approach |
|---|---|
| Clinical NLP, adverse event detection, EHR processing | Fine-tuned clinical model + ontology-based type validation |
| Drug discovery, literature mining, molecular research | GPT-4 with a refined type taxonomy + expert review |
| Production systems requiring both | Hybrid pipeline with post-processing validation layer |

### Architecture decisions shape what your system can see and what it calls things

The model you choose determines which *kinds* of entities your pipeline is capable of finding at all. A clinical model is blind to molecular biology; a general LLM is imprecise about clinical procedures. But even within each model's domain of strength, type label accuracy is a separate problem that neither model solves automatically. Match your model to your use case, then validate what it tells you.